# OCR LIB

In [1]:
from ocr_library.nn.pipelines.text_recognize_classify_pipeline import TextRecClsPipeline
from tqdm import tqdm
from PIL import Image
import numpy as np
from datetime import datetime


folder = "C:/razmetka/ML2"
with open(f'{folder}/rec_gt.txt', 'r', encoding='utf-8') as txt:
    valid_list = txt.readlines()

ocr = TextRecClsPipeline(
    infer_option='openvino',
    rec_model_path=r'C:\Users\pirat\OneDrive\Documents\Github\ml\models\ocr_rec\1\model.onnx',
    character_dict_path=r'C:\Users\pirat\OneDrive\Documents\Github\ml\models\ocr_rec_postproc\1\ru_dict_ext100124.txt'
)

good_preds = []
bad_preds = []
threshold = 0.95
for idx, line in tqdm(enumerate(valid_list)):
    try:
        path, _ = line.split('\t')
        image = Image.open(f'{folder}/{path}').convert('RGB')
        result = ocr(np.array([image]))

        if result[0][0][1] >= threshold:
            good_preds.append(f'{path}\t{result[0][0][0]}\n')
        else:
            bad_preds.append(f'{path}\t{result[0][0][0]}\n')
    except:
        print(line)
    # if idx==10:
    #     break
    
curr_time = str(datetime.now().strftime("%Y-%m-%d %H-%M-%S"))
with open(f'good_result_{int(threshold*100)}_{curr_time}.txt', 'w', encoding='utf-8') as txt:
    txt.write(''.join(good_preds))
with open(f'bad_result_{int(threshold*100)}_{curr_time}.txt', 'w', encoding='utf-8') as txt:
    txt.write(''.join(bad_preds))

44834it [13:00, 57.44it/s] 


In [ ]:
break

# SURYA

In [ ]:
from PIL import Image
from surya.foundation import FoundationPredictor
from surya.recognition import RecognitionPredictor
from surya.detection import DetectionPredictor
from tqdm import tqdm
import numpy as np
from datetime import datetime

foundation_predictor = FoundationPredictor()
recognition_predictor = RecognitionPredictor(foundation_predictor)
detection_predictor = DetectionPredictor()


folder = "C:/razmetka/ML2"
with open(f'{folder}/rec_gt.txt', 'r', encoding='utf-8') as txt:
    valid_list = txt.readlines()

good_preds = []
bad_preds = []
threshold = 0.95
for idx, line in tqdm(enumerate(valid_list)):
    try:
        path, _ = line.split('\t')
        image = Image.open(f'{folder}/{path}').convert('RGB')
        bboxes = [[[0.0, 0.0, image.size[0], image.size[1]]]]
        predictions = recognition_predictor([image], None, bboxes=bboxes)
        text = predictions[0].text_lines[0].text
        score = predictions[0].text_lines[0].confidence

        if score >= threshold:
            good_preds.append(f'{path}\t{text}\n')
        else:
            bad_preds.append(f'{path}\t{text}\n')
    except:
        print(line)
    if idx==1:
        break
    
curr_time = str(datetime.now().strftime("%Y-%m-%d %H-%M-%S"))
with open(f'surya_good_result_{int(threshold*100)}_{curr_time}.txt', 'w', encoding='utf-8') as txt:
    txt.write(''.join(good_preds))
with open(f'surya_bad_result_{int(threshold*100)}_{curr_time}.txt', 'w', encoding='utf-8') as txt:
    txt.write(''.join(bad_preds))

c:\Users\pirat\OneDrive\Documents\Github\dotsOCR\dots\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Recognizing Text: 100%|██████████| 1/1 [00:00<00:00,  6.10it/s]
1it [00:02,  2.32s/it]


In [ ]:
break

# CROSS

In [ ]:
from PIL import Image
from surya.foundation import FoundationPredictor
from surya.recognition import RecognitionPredictor
from surya.detection import DetectionPredictor
from tqdm import tqdm
import numpy as np
from datetime import datetime
from ocr_library.nn.pipelines.text_recognize_classify_pipeline import TextRecClsPipeline

foundation_predictor = FoundationPredictor()
recognition_predictor = RecognitionPredictor(foundation_predictor)
detection_predictor = DetectionPredictor()
ocr = TextRecClsPipeline(
    infer_option='openvino',
    rec_model_path=r'C:\Users\pirat\OneDrive\Documents\Github\ml\models\ocr_rec\1\model.onnx',
    character_dict_path=r'C:\Users\pirat\OneDrive\Documents\Github\ml\models\ocr_rec_postproc\1\ru_dict_ext100124.txt'
)

folder = "C:/razmetka/ML2"
with open(f'{folder}/rec_gt.txt', 'r', encoding='utf-8') as txt:
    valid_list = txt.readlines()

good_preds = []
bad_preds = []
threshold = 0.95
for idx, line in tqdm(enumerate(valid_list)):
    try:
        path, _ = line.split('\t')
        image = Image.open(f'{folder}/{path}').convert('RGB')
        
        bboxes = [[[0.0, 0.0, image.size[0], image.size[1]]]]
        surya_predictions = recognition_predictor([image], None, bboxes=bboxes)
        surya_text = predictions[0].text_lines[0].text
        surya_score = predictions[0].text_lines[0].confidence

        paddle_predictions = ocr(np.array([image]))
        paddle_text = result[0][0][0]
        paddle_score = result[0][0][1]

        if surya_text == paddle_text and paddle_text != "":
            good_preds.append(f'{path}\t{paddle_text}\n')
        else:
            bad_preds.append(f'{path}\t{paddle_text}\n')

    except:
        print(line)
    if idx==10:
        break
    
curr_time = str(datetime.now().strftime("%Y-%m-%d %H-%M-%S"))
with open(f'good_result_{int(threshold*100)}_{curr_time}.txt', 'w', encoding='utf-8') as txt:
    txt.write(''.join(good_preds))
with open(f'bad_result_{int(threshold*100)}_{curr_time}.txt', 'w', encoding='utf-8') as txt:
    txt.write(''.join(bad_preds))

# FULL

In [ ]:
from ocr_library.document_processing.utils import remove_control_characters, map_pdf_x_to_pix, map_pdf_y_to_pix
import cv2 as cv
import os

def extract_text_pdf(pdf_page, pages_or_dict):
    scale = pages_or_dict.iterator.get_scale_consider_max_len_side(
        pdf_page, pages_or_dict.dpi, pages_or_dict.max_len_side
    )
    frame = pdf_page.render(scale=scale, grayscale=pages_or_dict.grayscale).to_pil()

    frame = pages_or_dict._convert_color_mode(frame)
    frame = np.array(frame)
    if pages_or_dict.max_len_side is not None:
        max_width_height = max(frame.shape[:2])
        if max_width_height > pages_or_dict.max_len_side:
            coef = pages_or_dict.max_len_side / max_width_height
            frame = cv.resize(
                frame, None, fx=coef, fy=coef, interpolation=cv.INTER_AREA
            )
    numpy_image = frame

    layout_width, layout_height = pdf_page.get_size()
    image_height, image_width = numpy_image.shape[:2]
    textpage = pdf_page.get_textpage()
    result = []

    for obj in pdf_page.get_objects(filter = [1]): # filter = [1, 4]
        pdf_box = obj.get_pos()  # bbox объекта
        chars = []
        for i in range(textpage.count_chars()):
            cx0, cy0, cx1, cy1 = textpage.get_charbox(i)
            if cx0 >= pdf_box[0] and cx1 <= pdf_box[2] and cy0 >= pdf_box[1] and cy1 <= pdf_box[3]:
                ch = textpage.get_text_range(i, 1)
                chars.append((ch, (cx0, cy0, cx1, cy1)))

        if not chars:
            continue

        text = "".join(ch for ch, _ in chars)
        stripped = text.rstrip()
        if stripped.strip() == "":
            continue

        stripped = remove_control_characters(stripped)
        if stripped.replace(" ", "") == "":
            continue

        # оставляем только символы без хвостовых пробелов
        kept = chars[:len(stripped)]

        left = min(c[1][0] for c in kept)
        bottom = min(c[1][1] for c in kept)
        right = max(c[1][2] for c in kept)
        top = max(c[1][3] for c in kept)


        x_0 = int(map_pdf_x_to_pix(left, layout_width, image_width))
        y_0 = int(map_pdf_y_to_pix(top, layout_height, image_height))
        x_1 = int(map_pdf_x_to_pix(right, layout_width, image_width))
        y_1 = int(map_pdf_y_to_pix(bottom, layout_height, image_height))

        if x_1 - x_0 < 3 or y_1 - y_0 < 3:
            continue

        paddle_box = [
            [x_0, y_0],
            [x_1, y_0],
            [x_1, y_1],
            [x_0, y_1],
        ]

        result.append(
            [paddle_box, [stripped, 1.0]]
        )

    return result, numpy_image


def save_image(img_crop, img_count, images_one_folder, image_folder, num_digits):
    img_crop_folder = str(img_count // images_one_folder)
    os.makedirs(f"{image_folder}/{img_crop_folder}", exist_ok=True)
    image_save_path = f"{image_folder}/{img_crop_folder}/image_{img_count:0{num_digits}d}.webp"
    Image.fromarray(img_crop).save(image_save_path, 'WEBP')
    img_count += 1
    return img_count, image_save_path


def save_txt(image_folder, good_preds, bad_highscore_preds, bad_underscore_preds, threshold):
    curr_time = str(datetime.now().strftime("%Y-%m-%d %H-%M-%S"))
    with open(f'{image_folder}/good_result_{int(threshold*100)}_{curr_time}.txt', 'w', encoding='utf-8') as txt:
        txt.write(''.join(good_preds))
    with open(f'{image_folder}/bad_highscore_result_{int(threshold*100)}_{curr_time}.txt', 'w', encoding='utf-8') as txt:
        txt.write(''.join(bad_highscore_preds))
    with open(f'{image_folder}/bad_underscore_result_{int(threshold*100)}_{curr_time}.txt', 'w', encoding='utf-8') as txt:
        txt.write(''.join(bad_underscore_preds))

In [ ]:
from glob import glob
import os
from PIL import Image
from surya.foundation import FoundationPredictor
from surya.recognition import RecognitionPredictor
from surya.detection import DetectionPredictor
from tqdm import tqdm
import numpy as np
import random
from datetime import datetime
from ocr_library.document_processing import FileIterator
from ocr_library.nn.pipelines.text_recognize_classify_pipeline import TextRecClsPipeline



SCAN = ["pdf"]
IMAGE = ["jpg", "jpeg", "png", "tif", "tiff", "gif", "giff", "bmp", "webp"]
DOC = ["docx", "doc", "rtf", "odt", "pptx",
        "ppt", "odp", "xlsx", "xls", "ods"]
allowed_extensions = SCAN+IMAGE+DOC
folder_path = 'C:/Users/pirat/OneDrive/Documents/Github/ocr_markup'
filter_score = 0.95
matched_files = []
for ext in allowed_extensions:
    pattern = os.path.join(folder_path, f'*{ext}')
    matched_files.extend(glob(pattern))

good_preds = []
bad_highscore_preds = []
bad_underscore_preds = []
pdf_preds = []
image_folder = f"{folder_path}/images"
os.makedirs(image_folder, exist_ok=True)
img_count = 1
images_one_folder = 10000
random_padding=7
min_image_pix = 10
num_digits = len(str(images_one_folder))
# for file in matched_files:
for file_idx, file in enumerate([r'C:\Users\pirat\OneDrive\Documents\Github\rag-query-processor\тест.pdf']):
    pdf_or_images = FileIterator(file, grayscale=False, dpi=200, pdf_parsing=True, max_pages=None)
    for page_idx, page in enumerate(pdf_or_images):
        try:
            if pdf_or_images.iterator.pdf_available: 
                has_pdf = True
            else:
                has_pdf = False
        except:
            has_pdf = False

        if has_pdf:
            text_boxes, image = extract_text_pdf(page, pdf_or_images)
            for crop_idx, (box, text_conf) in enumerate(text_boxes):
                # img_crop = image.crop((box[0][0], box[0][1],box[2][0], box[2][1]))
                img_crop = image[
                    max(box[0][1]-random.randint(0, random_padding), 0):min(box[2][1]+random.randint(0, random_padding), image.shape[0]),
                    max(box[0][0]-random.randint(0, random_padding), 0):min(box[2][0]+random.randint(0, random_padding), image.shape[1])
                ]
                if img_crop.shape[0] <= min_image_pix or img_crop.shape[1] <= min_image_pix:
                    continue
                img_count, image_save_path = save_image(img_crop, img_count, images_one_folder, image_folder, num_digits)
                pdf_preds.append(f'{image_save_path}\t{text_conf[0]}\n')

        else:
            image = page
            surya_bboxes = [box.polygon for box in detection_predictor([image])[0].bboxes]
            
            for crop_idx, box in enumerate(surya_bboxes):
                surya_predictions = recognition_predictor([image], None, bboxes=[[[box[0][0], box[0][1], box[2][0], box[2][1]]]])
                surya_text = surya_predictions[0].text_lines[0].text
                surya_score = surya_predictions[0].text_lines[0].confidence

                img_crop = np.array(image)[box[0][1]:box[2][1], box[0][0]:box[2][0]]
                paddle_predictions = ocr(np.array([img_crop]))
                paddle_text = paddle_predictions[0][0][0]
                paddle_score = paddle_predictions[0][0][1]

                if img_crop.shape[0] <= min_image_pix or img_crop.shape[1] <= min_image_pix:
                    continue
                img_count, image_save_path = save_image(img_crop, img_count, images_one_folder, image_folder, num_digits)

                if surya_text == paddle_text and paddle_text != "":
                    good_preds.append(f'{image_save_path}\t{paddle_text}\n')
                else:
                    if paddle_score>=filter_score:
                        bad_highscore_preds.append(f'{image_save_path}\t{paddle_text}\n')
                    elif surya_score>=filter_score:
                        bad_highscore_preds.append(f'{image_save_path}\t{surya_text}\n')
                    else:
                        bad_underscore_preds.append(f'{image_save_path}\t{paddle_text}\n')
save_txt(good_preds, bad_highscore_preds, bad_underscore_preds, filter_score)

c:\Users\pirat\OneDrive\Documents\Github\dotsOCR\dots\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\pirat\OneDrive\Documents\Github\dotsOCR\dots\lib\site-packages\pypdfium2\_helpers\textpage.py:80: UserWarning: get_text_range() call with default params will be implicitly redirected to get_text_bounded()
  warnings.warn("get_text_range() call with default params will be implicitly redirected to get_text_bounded()")


In [ ]:
# import pytesseract

# res = pytesseract.image_to_string(image, lang='rus', config='--oem 3')
# res

'Как любитель Кларк занимался фантастической беллетристикой с 1930-х годов,\nпомещая свои рассказы в фэнзинах, в некоторых из которых являлся редактором. Как\nпрофессиональный писатель дебютировал в журнале Азюипатда Зсепсе Нсйопв мае\n1946 года рассказом «Спасательный отряд». В 1950-е годы увлёкся дайвингом и с 1954 года\nперебрался на постоянное место жительства на Цейлон, хотя много времени продолжал\nпроводить в Великобритании и США. В раннем творчестве Кларка выделялись романы «Город\nИ ЗВЁЗДЫ» И «КОНЕЦ ДЕТСТВА», ОТМЕЧЕННЫЕ НАГРАДАМИ И ВЫЗВАВШИЕ БОЛЬШОЕ\nВНИМАНИЕ КРИТИКОВ И ИССЛЕДОВАТЕЛЕЙ. ИЗВЕСТЕН ТАКЖЕ СОВМЕСТНОЙ РАБОТОЙ\nсо Стэнли Кубриком по созданию научно-фантастического фильма «Космическая одиссея\n2001 года» (1968). Параллельно Кларк писал роман «2001: Космическая одиссея»,\nвпоследствии дополненный тремя продолжениями.\n\nМного наград получили также романы «Свидание с Рамой» и «Фонтаны рая». В\n1975 году писатель получил ланкийское гражданство. Писательская активность при